# Reasoning Evaluation - Hindi

In [1]:
import os
import math
import json
import string
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
import sentencepiece as spm
from typing import Dict, List, Any

# ==========================================
# UPDATE THESE PATHS
# ==========================================
PRETRAINED_CHECKPOINT = "/kaggle/input/notebooks/srinivasu2005/train-hindi/checkpoints_hindi/best_model.pt"
FINETUNED_CHECKPOINT  = "/kaggle/input/notebooks/srinivasu2005/hindi-finetune/finetuned_model_hindi/best_finetuned_model.pt"
TEST_JSONL            = "/kaggle/input/datasets/srinivasu2005/finetune-hindi/test.jsonl"
TOKENIZER_PATH        = "/kaggle/input/notebooks/srinivasu2005/hindi-tokenizer/tokenizer/tokenizer.model"
OUTPUT_DIR            = "/kaggle/working/eval_hindi_finetuned"
os.makedirs(OUTPUT_DIR, exist_ok=True)

## Model Architecture

In [2]:
VOCAB_SIZE = 8000
D_FF = 1024

@dataclass
class TransformerConfig:
    vocab_size: int = VOCAB_SIZE; d_model: int = 512; n_layers: int = 8
    n_heads: int = 8; d_ff: int = D_FF; max_seq_len: int = 512
    dropout: float = 0.0; tie_weights: bool = True; rope_base: int = 10000
    pad_id: int = 0; bos_id: int = 2; eos_id: int = 3
    @property
    def d_k(self): return self.d_model // self.n_heads

def precompute_rope_frequencies(d_k, max_seq_len, base=10000, device=None):
    freq = torch.arange(0, d_k, 2, dtype=torch.float32, device=device)
    theta = 1.0 / (base ** (freq / d_k))
    pos = torch.arange(0, max_seq_len, dtype=torch.float32, device=device)
    angles = torch.outer(pos, theta)
    return torch.cos(angles), torch.sin(angles)

def apply_rope(x, cos, sin):
    x1, x2 = x[..., 0::2], x[..., 1::2]
    c = cos.unsqueeze(0).unsqueeze(0); s = sin.unsqueeze(0).unsqueeze(0)
    return torch.stack([x1*c - x2*s, x1*s + x2*c], dim=-1).flatten(-2)

class MultiHeadCausalSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n_heads, self.d_k, self.d_model = cfg.n_heads, cfg.d_k, cfg.d_model
        self.W_Q = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.W_K = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.W_V = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.W_O = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.attn_dropout = nn.Dropout(cfg.dropout)
        self.resid_dropout = nn.Dropout(cfg.dropout)
        cos, sin = precompute_rope_frequencies(cfg.d_k, cfg.max_seq_len, cfg.rope_base)
        self.register_buffer("cos_table", cos); self.register_buffer("sin_table", sin)
        self.register_buffer("causal_mask", torch.triu(torch.ones(cfg.max_seq_len, cfg.max_seq_len), diagonal=1).bool())
    def forward(self, x, return_attention=False):
        B, T, _ = x.shape
        Q = self.W_Q(x).view(B,T,self.n_heads,self.d_k).transpose(1,2)
        K = self.W_K(x).view(B,T,self.n_heads,self.d_k).transpose(1,2)
        V = self.W_V(x).view(B,T,self.n_heads,self.d_k).transpose(1,2)
        c, s = self.cos_table[:T], self.sin_table[:T]
        Q, K = apply_rope(Q,c,s), apply_rope(K,c,s)
        sc = torch.matmul(Q, K.transpose(-2,-1)) / math.sqrt(self.d_k)
        sc = sc.masked_fill(self.causal_mask[:T,:T].unsqueeze(0).unsqueeze(0), float('-inf'))
        aw = torch.softmax(sc, dim=-1)
        ctx = torch.matmul(self.attn_dropout(aw), V).transpose(1,2).contiguous().view(B,T,self.d_model)
        return (self.resid_dropout(self.W_O(ctx)), aw) if return_attention else (self.resid_dropout(self.W_O(ctx)), None)

class SwiGLUFeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.W_gate = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.W_up = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.W_down = nn.Linear(cfg.d_ff, cfg.d_model, bias=False)
        self.dropout = nn.Dropout(cfg.dropout)
    def forward(self, x): return self.dropout(self.W_down(F.silu(self.W_gate(x)) * self.W_up(x)))

class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model); self.attn = MultiHeadCausalSelfAttention(cfg)
        self.ln2 = nn.LayerNorm(cfg.d_model); self.ffn = SwiGLUFeedForward(cfg)
    def forward(self, x, return_attention=False):
        ao, aw = self.attn(self.ln1(x), return_attention); x = x + ao; x = x + self.ffn(self.ln2(x)); return x, aw

class DecoderOnlyTransformer(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.config = cfg
        self.embedding = nn.Embedding(cfg.vocab_size, cfg.d_model, padding_idx=cfg.pad_id)
        self.emb_dropout = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg.n_layers)])
        self.final_ln = nn.LayerNorm(cfg.d_model)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        if cfg.tie_weights: self.lm_head.weight = self.embedding.weight
    def forward(self, input_ids, targets=None, return_attention=False):
        x = self.emb_dropout(self.embedding(input_ids))
        all_attn = [] if return_attention else None
        for block in self.blocks:
            x, aw = block(x, return_attention)
            if return_attention and aw is not None: all_attn.append(aw)
        logits = self.lm_head(self.final_ln(x))
        result = {"logits": logits}
        if targets is not None:
            result["loss"] = F.cross_entropy(logits.view(-1, self.config.vocab_size), targets.view(-1), ignore_index=self.config.pad_id)
        if return_attention: result["attention_weights"] = all_attn
        return result
    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens=128, temperature=0.0, top_k=0, repetition_penalty=1.0):
        self.eval(); gen = input_ids
        for _ in range(max_new_tokens):
            ctx = gen if gen.shape[1] <= self.config.max_seq_len else gen[:, -self.config.max_seq_len:]
            logits = self.forward(ctx)["logits"][:, -1, :]
            if repetition_penalty != 1.0:
                for b in range(input_ids.size(0)):
                    for token_id in set(gen[b].tolist()):
                        if logits[b, token_id] < 0: logits[b, token_id] *= repetition_penalty
                        else: logits[b, token_id] /= repetition_penalty
            if temperature == 0.0: nt = logits.argmax(dim=-1, keepdim=True)
            else:
                logits = logits / temperature
                if top_k > 0:
                    v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                    logits[logits < v[:, [-1]]] = float('-inf')
                nt = torch.multinomial(torch.softmax(logits, dim=-1), 1)
            gen = torch.cat([gen, nt], dim=1)
            if (nt == self.config.eos_id).all(): break
        return gen

## Evaluation Logic

In [3]:
def load_model(checkpoint_path, device="cpu"):
    cfg = TransformerConfig(dropout=0.0)
    model = DecoderOnlyTransformer(cfg).to(device)
    if os.path.exists(checkpoint_path):
        try:
            state_dict = torch.load(checkpoint_path, map_location=device, weights_only=False)
        except TypeError:
            state_dict = torch.load(checkpoint_path, map_location=device)
        if isinstance(state_dict, dict) and "model_state_dict" in state_dict:
            state_dict = state_dict["model_state_dict"]
        model.load_state_dict(state_dict)
    else:
        print(f"Warning: Checkpoint not found at '{checkpoint_path}'. Evaluating randomly initialized model.")
    model.eval()
    return model

def normalize_text(text):
    return text.strip().strip(string.punctuation)

def extract_answer(generated_text, separator="उत्तर:"):
    if separator in generated_text:
        return generated_text.split(separator)[-1].strip()
    return generated_text.strip()

def categorize_failure(pred, gold, question):
    pred_norm, gold_norm = normalize_text(pred), normalize_text(gold)
    if not pred_norm: return "incomplete"
    q_words = set(question.split())
    if any(w in q_words for w in pred_norm.split()) and pred_norm != gold_norm:
        return "wrong_entity"
    opposites = [("हाँ", "नहीं"), ("दाएं", "बाएं"), ("ऊपर", "नीचे"), ("सत्य", "असत्य")]
    for w1, w2 in opposites:
        if (w1 in gold_norm and w2 in pred_norm) or (w2 in gold_norm and w1 in pred_norm):
            return "wrong_direction"
    if len(pred_norm) > len(gold_norm) * 3 or len(pred_norm) > 100:
        return "hallucination"
    return "other"

def run_evaluation():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if not os.path.exists(TOKENIZER_PATH):
        print("Tokenizer not found.")
        return
    sp = spm.SentencePieceProcessor(model_file=TOKENIZER_PATH)
    
    models = {
        "pretrained": load_model(PRETRAINED_CHECKPOINT, device),
        "finetuned": load_model(FINETUNED_CHECKPOINT, device)
    }
    
    examples = []
    if os.path.exists(TEST_JSONL):
        with open(TEST_JSONL, "r", encoding="utf-8") as f:
            for line in f:
                examples.append(json.loads(line))
    else:
        print(f"Warning: {TEST_JSONL} not found. Running with empty list.")
    
    results = {}
    
    for model_name, model in models.items():
        correct, total = 0, 0
        task_correct = {}
        task_total = {}
        success_examples = []
        failure_examples = []
        failure_counts = {"wrong_entity": 0, "wrong_direction": 0, "hallucination": 0, "incomplete": 0, "other": 0}
        
        for ex in examples:
            q = ex.get("question", "")
            if "उत्तर:" not in q: q += "\nउत्तर:"
            gold = ex.get("answer", "")
            q = ex.get("input", ex.get("question", ""))
            if "उत्तर:" not in q: q += " उत्तर:"
            gold = ex.get("target", ex.get("answer", ""))
            task_type = ex.get("task_type", "unknown")
            
            input_ids = [sp.bos_id()] + sp.encode(q)
            input_tensor = torch.tensor([input_ids], dtype=torch.long, device=device)
            
            out_tensor = model.generate(input_tensor, max_new_tokens=64, temperature=0.0)
            full_out = sp.decode(out_tensor[0].tolist())
            
            pred = extract_answer(full_out, "उत्तर:")
            is_correct = normalize_text(pred) == normalize_text(gold)
            
            total += 1
            task_total[task_type] = task_total.get(task_type, 0) + 1
            if is_correct:
                correct += 1
                task_correct[task_type] = task_correct.get(task_type, 0) + 1
                if len(success_examples) < 10:
                    success_examples.append({"question": q, "gold": gold, "pred": pred})
            else:
                cat = categorize_failure(pred, gold, q)
                failure_counts[cat] += 1
                if len(failure_examples) < 10:
                    failure_examples.append({"question": q, "gold": gold, "pred": pred, "category": cat})
        
        acc = correct / max(total, 1)
        per_task = {k: task_correct.get(k, 0) / task_total[k] for k in task_total}
        
        results[model_name] = {
            "accuracy": acc,
            "per_task_type": per_task,
            "success_examples": success_examples,
            "failure_examples": failure_examples,
            "failure_counts": failure_counts
        }
        print(f"--- {model_name.upper()} ---")
        print(f"Accuracy: {acc:.2%}")
        for k, v in per_task.items(): print(f"  {k}: {v:.2%}")
    
    with open(os.path.join(OUTPUT_DIR, "eval_results.json"), "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

if __name__ == "__main__":
    run_evaluation()

--- PRETRAINED ---
Accuracy: 0.00%
  multi_attribute: 0.00%
  multihop_chained: 0.00%
  transitive_ordering: 0.00%
  direct_comparison: 0.00%
  negated: 0.00%
  superlative: 0.00%
--- FINETUNED ---
Accuracy: 62.60%
  multi_attribute: 42.80%
  multihop_chained: 95.74%
  transitive_ordering: 96.88%
  direct_comparison: 46.22%
  negated: 67.42%
  superlative: 27.38%
